In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

from IPython.display import Video
from IPython.display import Audio as play_audio
from torchvision.utils import make_grid
from torchcodec.encoders import VideoEncoder

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

import matplotlib.pyplot as plt

import sys
sys.path.append('../../../')

from pathlib import Path

%load_ext autoreload
%autoreload 2
    
from computer_vision.video_mae.dataset.transforms import GroupMultiScaleCrop, Stack, ToTorchFormatTensor, GroupNormalize
from computer_vision.video_mae.dataset.masking_generator import TubeMaskingGenerator, RunningCellMaskingGenerator, Cell

In [2]:
import math
import numbers
import warnings
import random

import numpy as np
import torch
import torchvision
import torchvision.transforms.functional as F
from PIL import Image, ImageOps

In [4]:
from torchvision import transforms

class DataAugmentationForVideoMAEv2(object):

    def __init__(self, args, div=True, roll=False, input_mean=[0.485, 0.456, 0.406], input_std=[0.229, 0.224, 0.225], 
                 scales=[1., .875, .75, .66]):
        
        self.input_mean=input_mean
        self.input_std=input_std
        normalize=GroupNormalize(self.input_mean, self.input_std)
        self.train_augmentation=GroupMultiScaleCrop(args.input_size, scales)
        self.transform=transforms.Compose([self.train_augmentation, Stack(roll=roll), ToTorchFormatTensor(div=div), normalize])
        if args.mask_type=='tube': self.encoder_mask_map_generator=TubeMaskingGenerator(args.window_size, args.mask_ratio)
        else: raise NotImplementedError("Unsupported encoder masking strategy type")

        if args.decoder_mask_ratio>0.:
            if args.decoder_mask_type=='run_cell':
                self.decoder_mask_map_generator=RunningCellMaskingGenerator(args.window_size, args.decoder_mask_ratio)
            else: raise NotImplementedError("Unsupported decoder masking strategy type")

    def __call__(self, images):
        process_data, _=self.transform(images)
        encoder_mask_map=self.encoder_mask_map_generator()
        if hasattr(self, 'decoder_mask_map_generator'): decoder_mask_map=self.decoder_mask_map_generator()
        else: decoder_mask_map=1-encoder_mask_map
        return process_data, encoder_mask_map, decoder_mask_map

    def __repr__(self):
        repr="(DataAugmentationForVideoMAEv2,\n"
        repr+=f" transform={str(self.transform)},\n"
        repr+=f" Encoder Masking Generator= {str(self.encoder_mask_map_generator)},\n"
        if hasattr(self, 'decoder_mask_map_generator'): repr+=f" Decoder Masking Generator={self.decoder_mask_map_generator},\n"
        else: repr+=" Do not use decoder masking,\n"
        repr+=")"
        return repr

In [3]:
output_dirpath=Path('D:/results/ucf101/video_mae')
output_dirpath.mkdir(parents=True, exist_ok=True)
video_data=torch.load(output_dirpath/'extracted_video_data_numpy.pt', weights_only=False)
print(f'{type(video_data)=}, {video_data.shape}')
images=[Image.fromarray(video_data[vid,]) for vid in range(16)] # convert each frame to PIL.Image

input_mean = [0.485, 0.456, 0.406]
input_std = [0.229, 0.224, 0.225]

aug_op=GroupMultiScaleCrop(input_size=224, scales=[1, .875, .75, .66])
ret_imgs, label=aug_op((images, None))
aug_op=Stack(roll=False)
ret_imgs, label=aug_op((ret_imgs, label))
aug_op=ToTorchFormatTensor(div=True)
ret_imgs, label=aug_op((ret_imgs, label))
print(f"{type(ret_imgs)=}, {ret_imgs.shape=}, (min, max)=({ret_imgs.min().item()}, {ret_imgs.max().item()})")

aug_op=GroupNormalize(mean=input_mean, std=input_std)
ret_imgs, label=aug_op((ret_imgs, label))
print(f"{type(ret_imgs)=}, {ret_imgs.shape=}, (min, max)=({ret_imgs.min().item()}, {ret_imgs.max().item()})")

type(video_data)=<class 'numpy.ndarray'>, (16, 240, 320, 3)
type(ret_imgs)=<class 'torch.Tensor'>, ret_imgs.shape=torch.Size([48, 224, 224]), (min, max)=(0.0, 0.9254902005195618)
type(ret_imgs)=<class 'torch.Tensor'>, ret_imgs.shape=torch.Size([48, 224, 224]), (min, max)=(-2.1179039478302, 2.3088455200195312)
